# 🚦 Traffic Demand Prediction — Maximum R² Score
**Goal:** Predict `demand` (continuous) on `test.csv` using `train.csv`.  
**Metric:** `score = max(0, 100 × R²)`  
**Strategy:** Aggressive feature engineering → LightGBM / XGBoost ensemble → stacking meta-learner.


## 1 · Install & Import

In [ ]:
# ── Install extra packages (uncomment if running fresh on Colab) ──────────────
# !pip install lightgbm xgboost catboost optuna --quiet

import warnings, os, gc, time
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score
from sklearn.ensemble import (
    GradientBoostingRegressor, RandomForestRegressor,
    ExtraTreesRegressor, StackingRegressor
)
from sklearn.linear_model import Ridge, BayesianRidge
import lightgbm as lgb
import xgboost as xgb

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost not available – skipping.")

# ── GPU Detection & Setup ────────────────────────────────────────────────────
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    GPU_DEVICE_ID = torch.cuda.current_device()
else:
    print("⚠️  Running on CPU. For faster training, enable GPU in Kaggle settings.")
    GPU_DEVICE_ID = None

print("✅ All imports successful")


## 2 · Data Loading

In [ ]:
# ── Load Kaggle inputs (works in /kaggle/input or /kaggle/working) ──────────
def find_input_file(filename):
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in search_roots:
        if root.exists():
            matches = sorted(root.rglob(filename))
            if matches:
                return matches[0]
    return None

train_path = find_input_file("train.csv")
test_path = find_input_file("test.csv")
sample_path = find_input_file("sample_submission.csv")

if train_path is None or test_path is None or sample_path is None:
    raise FileNotFoundError("Could not find train.csv, test.csv, and sample_submission.csv in Kaggle input folders.")

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Sample: {sample_sub.shape}")
train.head()


## 3 · Quick Exploratory Analysis

In [ ]:
print("=== Train Info ===")
train.info()
print("\n=== Missing values (train) ===")
print(train.isnull().sum())
print("\n=== Missing values (test) ===")
print(test.isnull().sum())
print("\n=== Target distribution ===")
print(train['demand'].describe())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train['demand'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Demand Distribution')
np.log1p(train['demand']).hist(bins=50, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Log1p(Demand) Distribution')
plt.tight_layout(); plt.show()

# Correlation heat-map for numeric columns
num_cols = train.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(10, 6))
sns.heatmap(train[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Pearson Correlation'); plt.show()


## 4 · Feature Engineering

In [ ]:
def engineer_features(df, is_train=True):
    df = df.copy()

    # ── Timestamp features ──────────────────────────────────────────────────
    # 'timestamp' format: "0:00" / "0:15" / … / "23:45" (15-min buckets)
    df['timestamp'] = df['timestamp'].astype(str)
    df['hour']      = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute']    = df['timestamp'].str.split(':').str[1].astype(int)
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15   # 0-95 (15-min bucket)

    # ── Day features ────────────────────────────────────────────────────────
    df['day'] = df['day'].astype(int)
    df['day_mod7']  = df['day'] % 7          # proxy for day-of-week
    df['is_weekend'] = (df['day_mod7'] >= 5).astype(int)
    df['week_num']   = df['day'] // 7

    # ── Cyclic encoding (hour & time_slot carry circular periodicity) ───────
    df['hour_sin']  = np.sin(2 * np.pi * df['hour']      / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']      / 24)
    df['slot_sin']  = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['slot_cos']  = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['day_sin']   = np.sin(2 * np.pi * df['day_mod7']  / 7)
    df['day_cos']   = np.cos(2 * np.pi * df['day_mod7']  / 7)

    # ── Peak-hour flags ─────────────────────────────────────────────────────
    df['is_morning_peak'] = df['hour'].between(7,  9).astype(int)
    df['is_evening_peak'] = df['hour'].between(17, 19).astype(int)
    df['is_night']        = (df['hour'] < 6).astype(int)
    df['is_midday']       = df['hour'].between(11, 14).astype(int)

    # ── Geohash features ─────────────────────────────────────────────────────
    # Precision-4 prefix captures city-block area; precision-5 = street level
    df['geo3'] = df['geohash'].str[:3]
    df['geo4'] = df['geohash'].str[:4]
    df['geo5'] = df['geohash'].str[:5]
    df['geo_len'] = df['geohash'].str.len()

    # ── Road / infrastructure interactions ───────────────────────────────────
    df['RoadType']      = df['RoadType'].astype(str)
    df['NumberofLanes'] = pd.to_numeric(df['NumberofLanes'], errors='coerce').fillna(0)
    df['LargeVehicles'] = pd.to_numeric(df['LargeVehicles'], errors='coerce').fillna(0)
    df['Landmarks']     = pd.to_numeric(df['Landmarks'],     errors='coerce').fillna(0)

    df['lanes_road_interact'] = df['NumberofLanes'].astype(str) + '_' + df['RoadType']
    df['capacity_score']      = df['NumberofLanes'] * (1 + df['LargeVehicles'])

    # ── Weather / temperature ────────────────────────────────────────────────
    df['Weather']     = df['Weather'].astype(str)
    df['Temperature'] = pd.to_numeric(df['Temperature'], errors='coerce')
    df['temp_bucket'] = pd.cut(df['Temperature'], bins=10, labels=False).fillna(-1).astype(int)

    # ── Geo × time interactions ──────────────────────────────────────────────
    df['geo4_hour'] = df['geo4'] + '_h' + df['hour'].astype(str)
    df['geo4_slot'] = df['geo4'] + '_s' + df['time_slot'].astype(str)
    df['geo4_day']  = df['geo4'] + '_d' + df['day_mod7'].astype(str)

    return df

train = engineer_features(train, is_train=True)
test  = engineer_features(test,  is_train=False)

print("Train shape after feature eng:", train.shape)
print("Test  shape after feature eng:", test.shape)


## 5 · Rolling Lag Demand Features


In [ ]:
def add_safe_lag_features(train_df, test_df, target='demand'):
    required_train = {'Index', 'geohash', 'day', 'time_slot', target}
    required_test = {'Index', 'geohash', 'day', 'time_slot'}
    if not required_train.issubset(train_df.columns) or not required_test.issubset(test_df.columns):
        print('Safe lag features skipped: required sequential columns are missing.')
        return train_df, test_df

    train_idx = pd.to_numeric(train_df['Index'], errors='coerce')
    test_idx = pd.to_numeric(test_df['Index'], errors='coerce')
    train_seq = train_idx.notna().all() and train_idx.is_monotonic_increasing and train_idx.nunique() == len(train_idx)
    test_seq = test_idx.notna().all() and test_idx.is_monotonic_increasing and test_idx.nunique() == len(test_idx)
    if not (train_seq and test_seq):
        print('Safe lag features skipped: Index is not sequential.')
        return train_df, test_df

    train_out = train_df.copy()
    test_out = test_df.copy()
    train_out['_abs_slot'] = train_out['day'].astype(int) * 96 + train_out['time_slot'].astype(int)
    test_out['_abs_slot'] = test_out['day'].astype(int) * 96 + test_out['time_slot'].astype(int)

    global_mean_val = train_out[target].mean()

    lag_96_lookup = train_out[['geohash', '_abs_slot', target]].copy()
    lag_96_lookup['_abs_slot'] += 96
    lag_96_lookup = lag_96_lookup.groupby(['geohash', '_abs_slot'], as_index=False)[target].mean()
    lag_96_lookup = lag_96_lookup.rename(columns={target: 'demand_lag_96'})

    train_out = train_out.merge(lag_96_lookup, on=['geohash', '_abs_slot'], how='left')
    test_out = test_out.merge(lag_96_lookup, on=['geohash', '_abs_slot'], how='left')

    train_out['demand_lag_96_missing'] = train_out['demand_lag_96'].isna().astype(int)
    test_out['demand_lag_96_missing'] = test_out['demand_lag_96'].isna().astype(int)
    train_out['demand_lag_96'] = train_out['demand_lag_96'].fillna(global_mean_val)
    test_out['demand_lag_96'] = test_out['demand_lag_96'].fillna(global_mean_val)

    train_out = train_out.drop(columns=['_abs_slot'])
    test_out = test_out.drop(columns=['_abs_slot'])
    print("Safe lag features added: ['demand_lag_96', 'demand_lag_96_missing']")
    return train_out, test_out

train, test = add_safe_lag_features(train, test, target='demand')
print('Train shape after lag features:', train.shape)
print('Test  shape after lag features:', test.shape)


## 6 · Target Statistics Encoding (Leave-One-Out on Train)

In [ ]:
from sklearn.model_selection import KFold

TARGET = 'demand'
SMOOTHING = 30   # smoothing factor for global mean regularisation

# Columns to target-encode
TE_COLS = ['geohash', 'geo3', 'geo4', 'geo5',
           'RoadType', 'Weather',
           'geo4_hour', 'geo4_slot', 'geo4_day',
           'lanes_road_interact']

global_mean = train[TARGET].mean()

def target_encode_cv(train_df, test_df, col, target, n_splits=5, smoothing=SMOOTHING):
    """
    Out-of-fold mean encoding on train; full-train smoothed mean on test.
    Prevents data leakage.
    """
    oof_te = np.zeros(len(train_df))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    global_mean_val = train_df[target].mean()

    for tr_idx, val_idx in kf.split(train_df):
        tr_fold = train_df.iloc[tr_idx]
        stats   = tr_fold.groupby(col)[target].agg(['mean', 'count'])
        stats['smooth'] = (stats['mean'] * stats['count'] + global_mean_val * smoothing) /                           (stats['count'] + smoothing)
        oof_te[val_idx] = train_df.iloc[val_idx][col].map(stats['smooth']).fillna(global_mean_val)

    # Test encoding: use full training set stats
    full_stats   = train_df.groupby(col)[target].agg(['mean', 'count'])
    full_stats['smooth'] = (full_stats['mean'] * full_stats['count'] + global_mean_val * smoothing) /                            (full_stats['count'] + smoothing)
    test_te = test_df[col].map(full_stats['smooth']).fillna(global_mean_val)
    return oof_te, test_te.values

for col in TE_COLS:
    tr_enc, te_enc = target_encode_cv(train, test, col, TARGET)
    train[col + '_te'] = tr_enc
    test[col  + '_te'] = te_enc
    print(f"Encoded: {col}")

print("\nEncoding done ✅")


## 7 · Geo-Aggregate & Time-Aggregate Features

In [ ]:
# ── Geo × time aggregate features (OOF-safe, same CV as target encoding) ──────
# Computing raw group means from the full train target leaks label info into
# the validation folds. We use the same target_encode_cv() as Cell 13 to get
# leak-free OOF values for train and smoothed-full-train values for test.

GEO_AGG_COLS = ['geo4_slot', 'geo4_day']   # already created in engineer_features

for col in GEO_AGG_COLS:
    tr_enc, te_enc = target_encode_cv(train, test, col, TARGET,
                                      n_splits=5, smoothing=SMOOTHING)
    suffix = '_agg_mean'   # distinct from the _te suffix used in Cell 13
    train[col + suffix] = tr_enc
    test[col  + suffix] = te_enc
    print(f"Geo-agg encoded (OOF): {col}")

print("\nGeo-aggregate features added ✅")
print("Train shape:", train.shape)


## 8 · Label Encode Remaining Categoricals

In [ ]:
# Columns not target-encoded but still categorical → ordinal encode
CAT_LABEL = ['geohash', 'geo3', 'geo4', 'geo5',
             'RoadType', 'Weather',
             'geo4_hour', 'geo4_slot', 'geo4_day',
             'lanes_road_interact']

combined = pd.concat([train, test], axis=0, ignore_index=True)

for col in CAT_LABEL:
    if combined[col].dtype == object:
        combined[col] = LabelEncoder().fit_transform(combined[col].astype(str))

train_n = len(train)
train = combined.iloc[:train_n].copy()
test  = combined.iloc[train_n:].copy()

print("Label encoding done ✅")


## 8 · Define Final Feature Set

In [ ]:
DROP_COLS = ['Index', 'demand', 'timestamp']

FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

X_train = train[FEATURE_COLS]
y_train = train[TARGET]
X_test  = test[FEATURE_COLS]

# Fill any remaining NaNs
X_train = X_train.fillna(-999)
X_test  = X_test.fillna(-999)

print(f"Features : {len(FEATURE_COLS)}")
print(f"X_train  : {X_train.shape}")
print(f"X_test   : {X_test.shape}")
print(FEATURE_COLS)


## 9 · Cross-Validation Utility

In [ ]:
N_SPLITS   = 5
RANDOM_SEED = 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

def run_kfold(model, X, y, X_te, label='Model', use_log=False):
    """
    Runs KFold, returns (oof_predictions, test_predictions, cv_r2_scores).
    use_log: train on log1p(y), invert with expm1.
    """
    oof   = np.zeros(len(X))
    preds = np.zeros(len(X_te))
    scores = []

    _y = np.log1p(y) if use_log else y.values

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = _y[tr_idx],     _y[val_idx]

        model.fit(X_tr, y_tr)
        val_pred = model.predict(X_val)
        te_pred  = model.predict(X_te)

        if use_log:
            val_pred = np.expm1(val_pred)
            te_pred  = np.expm1(te_pred)
            y_val    = np.expm1(y_val)

        val_pred = np.clip(val_pred, 0, None)
        te_pred  = np.clip(te_pred,  0, None)

        r2 = r2_score(y_val, val_pred)
        scores.append(r2)
        oof[val_idx] = val_pred
        preds += te_pred / N_SPLITS

        print(f"  Fold {fold}: R²={r2:.5f}")

    print(f"[{label}] Mean R²={np.mean(scores):.5f} ± {np.std(scores):.5f}\n")
    return oof, preds, scores


## 10 · LightGBM (Primary Model)

In [ ]:
lgb_params = dict(
    objective        = 'regression_l1',   # MAE objective → robust to outliers
    metric           = 'rmse',
    learning_rate    = 0.03,
    n_estimators     = 3000,
    num_leaves       = 127,
    max_depth        = -1,
    min_child_samples= 20,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    n_jobs           = -1,
    random_state     = RANDOM_SEED,
    verbose          = -1,
)

# ── GPU acceleration for LightGBM ────────────────────────────────────────────
if DEVICE == "cuda":
    lgb_params['device_type'] = 'gpu'
    lgb_params['gpu_device_id'] = GPU_DEVICE_ID

lgb_model = lgb.LGBMRegressor(**lgb_params)
oof_lgb, pred_lgb, scores_lgb = run_kfold(lgb_model, X_train, y_train, X_test, label='LightGBM', use_log=True)


## 11 · XGBoost

In [ ]:
xgb_params = dict(
    objective        = 'reg:squarederror',
    learning_rate    = 0.03,
    n_estimators     = 2000,
    max_depth        = 7,
    min_child_weight = 5,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    gamma            = 0.1,
    reg_alpha        = 0.05,
    reg_lambda       = 1.5,
    random_state     = RANDOM_SEED,
    verbosity        = 0,
    # XGBoost >= 3.1: use device="cuda" instead of tree_method="gpu_hist" + gpu_id
    device           = "cuda" if DEVICE == "cuda" else "cpu",
)

xgb_model = xgb.XGBRegressor(**xgb_params)
oof_xgb, pred_xgb, scores_xgb = run_kfold(xgb_model, X_train, y_train, X_test, label='XGBoost', use_log=True)


## 12 · CatBoost (if available)

In [ ]:
if HAS_CATBOOST:
    # ── GPU acceleration for CatBoost ───────────────────────────────────────
    cb_gpu_params = dict(
        task_type='GPU' if DEVICE == 'cuda' else 'CPU',
        devices=str(GPU_DEVICE_ID) if DEVICE == 'cuda' else None,
    ) if DEVICE == 'cuda' else {}
    
    cb_model = CatBoostRegressor(
        loss_function  = 'MAE',
        iterations     = 3000,
        learning_rate  = 0.03,
        depth          = 8,
        l2_leaf_reg    = 3,
        random_seed    = RANDOM_SEED,
        verbose        = 0,
        **cb_gpu_params
    )
    oof_cb, pred_cb, scores_cb = run_kfold(cb_model, X_train, y_train, X_test, label='CatBoost', use_log=True)
else:
    print("CatBoost skipped. Using LGB + XGB ensemble only.")
    oof_cb, pred_cb = oof_lgb.copy(), pred_lgb.copy()   # fallback: duplicate LGB


## 13 · Weighted Blend (OOF-optimised)

In [ ]:
from scipy.optimize import minimize

def blend_r2(weights, oofs, y):
    w = np.array(weights)
    w = np.clip(w, 0, None)
    w /= w.sum()
    blended = sum(w[i] * oofs[i] for i in range(len(oofs)))
    return -r2_score(y, np.clip(blended, 0, None))

oof_stack = [oof_lgb, oof_xgb, oof_cb]
pred_stack = [pred_lgb, pred_xgb, pred_cb]

result = minimize(
    blend_r2,
    x0=[1/3, 1/3, 1/3],
    args=(oof_stack, y_train),
    method='Nelder-Mead',
    options={'maxiter': 5000}
)
opt_w = np.clip(result.x, 0, None)
opt_w /= opt_w.sum()
print(f"Optimal weights → LGB: {opt_w[0]:.4f} | XGB: {opt_w[1]:.4f} | CB: {opt_w[2]:.4f}")

oof_blend  = sum(opt_w[i] * oof_stack[i]  for i in range(3))
pred_blend = sum(opt_w[i] * pred_stack[i] for i in range(3))
blend_r2_score = r2_score(y_train, np.clip(oof_blend, 0, None))
print(f"\nBlend OOF R² = {blend_r2_score:.5f}  (hackathon score ≈ {100*max(0,blend_r2_score):.2f})")


## 15 · Stacking Meta-Learner

In [ ]:
# Build level-1 feature matrix from OOF predictions
oof_matrix  = np.column_stack(oof_stack)    # shape (n_train, 3)
test_matrix = np.column_stack(pred_stack)   # shape (n_test,  3)

# ── Meta-learner: Ridge regression (fast, generalises well) ───────────────────
meta = Ridge(alpha=1.0)
meta.fit(oof_matrix, y_train)
meta_pred_oof  = meta.predict(oof_matrix)
meta_pred_test = meta.predict(test_matrix)

meta_pred_oof  = np.clip(meta_pred_oof,  0, None)
meta_pred_test = np.clip(meta_pred_test, 0, None)

stacking_r2 = r2_score(y_train, meta_pred_oof)
print(f"Stacking OOF R² = {stacking_r2:.5f}  (hackathon score ≈ {100*max(0,stacking_r2):.2f})")


## 15 · Choose Best Predictions

In [ ]:
# Compare all approaches and pick winner
scores_dict = {
    'LightGBM':  r2_score(y_train, np.clip(oof_lgb,   0, None)),
    'XGBoost':   r2_score(y_train, np.clip(oof_xgb,   0, None)),
    'CatBoost':  r2_score(y_train, np.clip(oof_cb,    0, None)),
    'Blend':     blend_r2_score,
    'Stacking':  stacking_r2,
}

print("=== OOF R² Comparison ===")
for name, sc in sorted(scores_dict.items(), key=lambda x: -x[1]):
    print(f"  {name:12s}: {sc:.5f}  (score≈{100*max(0,sc):.2f})")

best_model = max(scores_dict, key=scores_dict.get)
print(f"\n✅ Best model: {best_model}")

if best_model == 'LightGBM':
    final_pred = pred_lgb
elif best_model == 'XGBoost':
    final_pred = pred_xgb
elif best_model == 'CatBoost':
    final_pred = pred_cb
elif best_model == 'Blend':
    final_pred = pred_blend
else:
    final_pred = meta_pred_test

final_pred = np.clip(final_pred, 0, None)
print(f"Predictions summary: min={final_pred.min():.5f}  max={final_pred.max():.5f}  mean={final_pred.mean():.5f}")


## 17 · Feature Importance (LightGBM)

In [ ]:
lgb_model.fit(X_train, np.log1p(y_train))   # retrain on full train for importance

imp_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(30)

plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', data=imp_df, palette='viridis')
plt.title('Top-30 Feature Importances (LightGBM gain)')
plt.tight_layout(); plt.show()
print(imp_df.to_string(index=False))


## 17 · Optuna Hyperparameter Tuning (Optional – Run if Time Permits)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = dict(
        objective        = 'regression_l1',
        metric           = 'rmse',
        verbose          = -1,
        n_jobs           = -1,
        random_state     = RANDOM_SEED,
        learning_rate    = trial.suggest_float('lr',           0.01, 0.1,  log=True),
        n_estimators     = trial.suggest_int  ('n_est',        500,  3000),
        num_leaves       = trial.suggest_int  ('num_leaves',   31,   255),
        min_child_samples= trial.suggest_int  ('min_child',    5,    50),
        subsample        = trial.suggest_float('subsample',    0.5,  1.0),
        colsample_bytree = trial.suggest_float('colsample',    0.5,  1.0),
        reg_alpha        = trial.suggest_float('reg_alpha',    1e-3, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda',   1e-3, 10.0, log=True),
    )
    
    # GPU acceleration for Optuna trials
    if DEVICE == "cuda":
        params['device_type'] = 'gpu'
        params['gpu_device_id'] = GPU_DEVICE_ID
    
    model = lgb.LGBMRegressor(**params)
    cv_scores = []
    for tr_idx, val_idx in kf.split(X_train):
        model.fit(X_train.iloc[tr_idx], np.log1p(y_train.iloc[tr_idx]))
        pred = np.expm1(model.predict(X_train.iloc[val_idx]))
        cv_scores.append(r2_score(y_train.iloc[val_idx], np.clip(pred, 0, None)))
    return np.mean(cv_scores)

print(f"🔧 Running Optuna HPO on {DEVICE.upper()}...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n✅ Optuna study complete!")
print("Best params:", study.best_params)
print(f"Best R² (OOF): {study.best_value:.5f}")

# ── Auto-plug best params back into lgb_params and retrain ───────────────────
bp = study.best_params
lgb_params = dict(
    objective        = 'regression_l1',
    metric           = 'rmse',
    verbose          = -1,
    n_jobs           = -1,
    random_state     = RANDOM_SEED,
    learning_rate    = bp['lr'],
    n_estimators     = bp['n_est'],
    num_leaves       = bp['num_leaves'],
    max_depth        = -1,
    min_child_samples= bp['min_child'],
    subsample        = bp['subsample'],
    colsample_bytree = bp['colsample'],
    reg_alpha        = bp['reg_alpha'],
    reg_lambda       = bp['reg_lambda'],
)

# GPU acceleration for tuned LGB
if DEVICE == "cuda":
    lgb_params['device_type'] = 'gpu'
    lgb_params['gpu_device_id'] = GPU_DEVICE_ID

print("\n▶ Retraining LightGBM with tuned hyperparameters...")
lgb_model = lgb.LGBMRegressor(**lgb_params)
oof_lgb, pred_lgb, scores_lgb = run_kfold(lgb_model, X_train, y_train, X_test, label='LightGBM (tuned)', use_log=True)

# Refresh the blend and stacking with the updated LGB predictions
oof_stack[0]  = oof_lgb
pred_stack[0] = pred_lgb

result2 = minimize(
    blend_r2,
    x0=[1/3, 1/3, 1/3],
    args=(oof_stack, y_train),
    method='Nelder-Mead',
    options={'maxiter': 5000}
)
opt_w2 = np.clip(result2.x, 0, None)
opt_w2 /= opt_w2.sum()
print(f"\nUpdated blend weights → LGB: {opt_w2[0]:.4f} | XGB: {opt_w2[1]:.4f} | CB: {opt_w2[2]:.4f}")

oof_blend  = sum(opt_w2[i] * oof_stack[i]  for i in range(3))
pred_blend = sum(opt_w2[i] * pred_stack[i] for i in range(3))
blend_r2_score = r2_score(y_train, np.clip(oof_blend, 0, None))

oof_matrix2  = np.column_stack(oof_stack)
test_matrix2 = np.column_stack(pred_stack)
meta.fit(oof_matrix2, y_train)
meta_pred_oof  = np.clip(meta.predict(oof_matrix2),  0, None)
meta_pred_test = np.clip(meta.predict(test_matrix2), 0, None)
stacking_r2    = r2_score(y_train, meta_pred_oof)

print(f"Blend OOF R² = {blend_r2_score:.5f}  (score ≈ {100*max(0,blend_r2_score):.2f})")
print(f"Stacking OOF R² = {stacking_r2:.5f}  (score ≈ {100*max(0,stacking_r2):.2f})")


## 19 · Generate Submission File

In [ ]:
submission = pd.DataFrame({
    'Index':  test['Index'].astype(int),
    'demand': final_pred
})

# Validate against sample submission format
assert submission.shape[1] == 2,            "❌ Wrong number of columns"
assert len(submission) == 41778,            "❌ Wrong number of rows"
assert list(submission.columns) == ['Index','demand'], "❌ Column name mismatch"
assert (submission['demand'] >= 0).all(),   "⚠️  Negative demand values clipped to 0"

submission_path = '/kaggle/working/submission.csv'
submission.to_csv(submission_path, index=False)

print(f"✅ Submission saved → {submission_path}")
print(f"   Shape  : {submission.shape}")
print(f"   demand : min={submission['demand'].min():.5f}  max={submission['demand'].max():.5f}  mean={submission['demand'].mean():.5f}")
submission.head(10)


## 20 · Submission File


In [ ]:
print("Submission file is ready in /kaggle/working/submission.csv")
